# LS 7-to-1 |Y⟩ State Distillation

Steane 7-to-1 magic state distillation via Lattice Surgery on the unrotated surface code.

## Protocol overview

Five surface code patches arranged in a cross layout:

```
W1  W3
W2  W4  ← output patch
W5      ← ancilla (reused)
```

- **W1, W2, W3, W5**: |Y⟩ inputs, prepared with noisy fold-transversal S gate
- **W4**: |+⟩_L output (fault-tolerant |Y⟩ after distillation)
- Four sequential ZZZZ product measurements (lattice surgery): {W1,W2,W3,W5}, {W1,W2,W4,W5}, {W1,W3,W4,W5}, {W2,W3,W4,W5}
- Final readout: noiseless S†_L on W4 → transversal MX

## |Y⟩ state preparation method

**`fold_transversal_s`** — NOT state injection (corner/middle qubit protocol).

Physical operations: RX all data qubits → transversal S (Clifford phase gate).  
This prepares |Y⟩ = S_L|+⟩_L. The code distance d partially protects against errors in the S gate itself.

**Why not corner injection?** Corner injection injects at a single physical qubit, so protection is only achieved through post-selection. The fold-transversal S applies across all d² data qubits, allowing syndromes to detect errors during the subsequent SE rounds. In practice, fold-transversal S achieves lower p_in for the same physical error rate.

## p_in calibration

p_in = logical error rate of one noisy |Y⟩ preparation.  
Calibration circuit: RX → noisy S → SE(rounds) → **noiseless S†** → MX.  
The noiseless S†+MX is fault-tolerant readout; only the preparation noise contributes to LER.

## p_out measurement

Run the full 5-patch distillation circuit with injection-only noise.  
Post-select on W1, W2, W3 syndrome observables (corrected == 0 after decoding).  
Target = W4 observable → p_out.

Theory: p_out ≈ 7 · p_in³  (third-order Steane distillation)

Code: Unrotated Surface Code, d=3, rounds=1 (quick demo; use rounds=d for paper benchmark).

In [ ]:
import sys
from pathlib import Path
import io, contextlib

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from benchmarks.logical_circuits.distillation.ls_7to1.LS_distillation_7_to_1 import (
    build_distillation_circuit,
    inject_noise,
    estimate_p_in,
    run_simulation,
    _LS_MAGIC_NAMES,
)
from lightstim.simulation.observable_analysis import (
    build_obs_patch_matrix,
    identify_distillation_observables,
)
from lightstim.simulation.decoder_backend import SimulationPipeline, DecoderConfig

In [ ]:
d = 3
rounds = 1   # use rounds=d for paper benchmark quality

## 1. Build Circuit

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    circuit, circuit_info, system = build_distillation_circuit(d, rounds)
print(f"d={d}: {circuit_info['num_qubits']} qubits, {circuit_info['num_detectors']} detectors, {circuit_info['num_observables']} observables")
print(f"Corridor internal width: {circuit_info['corridor_internal_width']}")

## 2. Noiseless Sanity Check

In [ ]:
import numpy as np
dets, obs = circuit.compile_detector_sampler().sample(shots=500, separate_observables=True)
print(f"Noiseless: det errors={np.any(dets)}, obs errors={np.any(obs)}")

## 3. Observable Analysis

Which observables correspond to the W4 output vs. post-select?  
Uses GF(2) elimination: find a linear combination of raw observables so that one targets W4 and the rest target the post-select patches (W1, W2, W3).

In [ ]:
matrix, patch_names = build_obs_patch_matrix(circuit, system)
print(f"Raw obs-to-patch matrix ({matrix.shape[0]} obs × {len(patch_names)} patches):")
print(f"Patches: {patch_names}")
for i in range(matrix.shape[0]):
    print(f"  L{i}: {list(matrix[i])}")

_, target_obs, ps_obs = identify_distillation_observables(matrix, patch_names, ["W4"])
print(f"\nAfter GF(2) elimination:")
print(f"  Target obs (W4 output): {target_obs}")
print(f"  Post-select obs (W1,W2,W3 syndromes): {ps_obs}")

## 4. p_in Calibration

Estimate the effective logical input infidelity of one noisy |Y⟩ prepared with `fold_transversal_s`.

Noise model: injection-only → Z_ERROR(p_injected) after each data qubit RX reset.  
This simulates an imperfect S gate on |+⟩ data qubits.

In [ ]:
p_injected_values = [1e-3, 5e-3, 2e-2]
p_in_values = {}

for p_inj in p_injected_values:
    p_in = estimate_p_in(d, rounds, p_injected=p_inj, max_shots=500_000, max_errors=50, batch_size=5_000)
    p_in_values[p_inj] = p_in
    print(f"p_injected={p_inj:.0e}  →  p_in={p_in:.3e}")

## 5. Distillation Simulation (injection-only noise)

Run the full 5-patch circuit with injection-only noise, post-select on W1/W2/W3, measure W4 LER.

In [ ]:
magic_qubits = {q for q, owner in system.index_to_owner_map.items()
                if owner in _LS_MAGIC_NAMES}
magic_data_qubits = magic_qubits & system.data_indices

results = []
for p_inj in p_injected_values:
    print(f"\np_injected={p_inj:.0e}  (p_in≈{p_in_values[p_inj]:.2e})")
    stats = run_simulation(
        circuit, magic_qubits, p=0.0, p_injected=p_inj, mode="injection",
        ps_obs=ps_obs, target_obs=target_obs,
        decoder_name="pymatching",
        max_shots=500_000, max_errors=30, batch_size=5_000,
        data_indices=magic_data_qubits,
    )
    results.append({
        'p_inj': p_inj,
        'p_in': p_in_values[p_inj],
        'p_out': stats.logical_error_rate,
        'ps_rate': stats.post_selection_rate,
        'shots': stats.shots,
        'errors': stats.errors,
    })
    print(f"  p_out={stats.logical_error_rate:.2e}  PS_rate={stats.post_selection_rate:.2f}  ({stats.errors}/{stats.shots:,})")

## 6. P_out vs P_in Plot

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))

p_in_arr = [r['p_in'] for r in results if r['errors'] >= 5]
p_out_arr = [r['p_out'] for r in results if r['errors'] >= 5]

if p_in_arr:
    ax.loglog(p_in_arr, p_out_arr, 'o-', color='steelblue', lw=2, ms=8,
              markeredgecolor='k', markeredgewidth=0.4, label=f'd={d}')
    pin_th = np.logspace(np.log10(min(p_in_arr)) - 0.5, np.log10(max(p_in_arr)) + 0.3, 200)
    ax.loglog(pin_th, 7 * pin_th**3, 'k--', lw=1.8, label=r'$7\,P_{\rm in}^3$')

ax.set_xlabel(r'$P_{\rm in}$', fontsize=13)
ax.set_ylabel(r'$P_{\rm out}$', fontsize=13)
ax.set_title(f'LS 7-to-1 Distillation — d={d}', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout()
plt.show()